# DIRISA SDC 2026: Electoral participation in Mpumalanga

**Problem statement.** Turnout in Mpumalanga's local government elections fell from 56.4% in 2016 to 42.8% in 2021 (IEC results, team calculation). Ahead of the 4 November 2026 elections, only about half of Mpumalanga's eligible 18-29-year-olds are on the voters' roll. At the same time, the Auditor-General keeps reporting weak financial management in several of the province's municipalities. These pieces of evidence sit in separate IEC, Census and Auditor-General sources, so it is hard to see *where* participation is at risk and *why*.

**Our question.** *Which Mpumalanga municipalities and voting districts are most at risk of low turnout in 2026, how much of that risk comes from registration gaps versus turnout gaps, where does a young voters' roll overlap with historically low turnout, and how does municipal performance relate to participation?*

**Who it is for:** the IEC and civil society (targeting voter education), journalists (which areas to watch), and municipalities and parties (where registration isn't turning into votes).

| Part | Method | Answers |
|---|---|---|
| A | Turnout forecast (machine learning, backtested on 2016 and 2021) | Which areas are most at risk of low turnout in 2026 |
| B | Registration gap vs turnout gap (calculated) | Is the problem registration, or voting? |
| C | Participation profiles and priority list | Where youth, low turnout and weak municipal performance overlap |

**Notebook map.** Each stage is completed and checked before the next one is added.

| Stage | Purpose | Status |
|---|---|---|
| 1 | Environment, metadata, source checks and raw ingestion | Ready |
| 2 | Mpumalanga election panel (voting district x election x ballot, 2000-2021), 2021 boundaries, reconciliation with IEC | Ready |
| 3 | Census 2022 indicators, registration denominators and Auditor-General audit outcomes | Ready |
| 4 | Exploratory analysis and feature engineering | Ready |
| 5 | Models: turnout forecast, gap breakdown, participation profiles | Next |
| 6 | Dashboard | To do |

**Scope:** Mpumalanga only (team decision), 17 local municipalities on 2021 boundaries. Data rules are in `metadata/README.md`. Raw files are never modified, and every derived table goes to `derived/`.

## Stage 1: Environment, metadata and raw ingestion

Checks the environment, loads the metadata that governs every input, confirms every source file exists, records checksums, and loads the core raw tables at their original grain. Nothing is cleaned, joined or modelled here.

In [ ]:
# =====================================================================================
# 1.1 Environment and configuration
# -------------------------------------------------------------------------------------
# WHAT:  Imports the libraries, sets the folders, and fixes the project scope.
# WHY:   Everything later depends on these settings, and the kernel check stops the
#        notebook early if the wrong Python is selected (a problem we hit before).
# LOOK FOR: "pandas: 3.0.6". If the cell errors, pick .venv\Scripts\python.exe as kernel.
# =====================================================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

RANDOM_SEED = 2026
PROJECT_ROOT = Path.cwd()
METADATA_DIR = PROJECT_ROOT / 'metadata'
DATASETS_DIR = PROJECT_ROOT / 'DATASETS'
DERIVED_DIR = PROJECT_ROOT / 'derived'
DERIVED_DIR.mkdir(exist_ok=True)

# Team decision: the analysis covers Mpumalanga only (17 municipalities, 2021 boundaries)
SCOPE_PROVINCE = 'Mpumalanga'
SCOPE_CENSUS_PROVINCE_CODE = 8          # Stats SA province code for Mpumalanga
SCOPE_MUNICIPALITIES = 17

print(f'Project root : {PROJECT_ROOT}')
print(f'Python       : {sys.version.split()[0]} | pandas: {pd.__version__} | numpy: {np.__version__}')
print(f'Scope        : {SCOPE_PROVINCE} ({SCOPE_MUNICIPALITIES} municipalities)')
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / '.venv').resolve(), \
    f'Wrong kernel: {sys.prefix}. Select .venv\\Scripts\\python.exe in the kernel picker.'

In [ ]:
# =====================================================================================
# 1.2 Source files
# -------------------------------------------------------------------------------------
# WHAT:  Lists every file the pipeline reads, and checks they all exist.
# WHY:   One place to see all inputs. If a file is missing, we find out now, not halfway
#        through the notebook.
# NOTE:  The 2000/2006 results come from the baseline file, which covers all provinces.
#        It must stay whole because Bushbuckridge (MP325) was counted under Limpopo in 2000.
# =====================================================================================
SOURCES = {
    # metadata
    'metadata_source_registry': METADATA_DIR / 'source_registry.csv',
    'metadata_data_dictionary': METADATA_DIR / 'data_dictionary.csv',
    'metadata_geo_crosswalk':   METADATA_DIR / 'geography_crosswalk.csv',
    'metadata_census_codebook': METADATA_DIR / 'census2022_household_codebook_derived.csv',
    'metadata_readme':          METADATA_DIR / 'README.md',
    # IEC election results
    'baseline_lge_master':      PROJECT_ROOT / 'LGE_All_Sources_Master_721327_Rows.csv',   # 2000 and 2006 (and MP 2011-2021 for cross-checks)
    'lge_2011_mp':              DATASETS_DIR / 'LGE2011' / 'MP_2011.csv',
    'lge_2016_mp':              DATASETS_DIR / 'LGE2016_MP' / 'MP_2016.csv',
    'lge_2021_mp':              DATASETS_DIR / 'LGE2021_MP' / 'MP_2021.csv',
    'iec_official_turnout_dir': DATASETS_DIR / 'IEC_official_turnout',                     # official MP turnout reports (checks)
    'npe_turnout_mp':           DATASETS_DIR / 'voter_turnout' / 'Mpumalanga_Voter_Turnout_2004-2024.csv',  # provincial elections, per municipality
    # registration
    'voters_2026_report':       DATASETS_DIR / '2026 Registered voters.xlsx',
    'youth_registration_2026':  DATASETS_DIR / 'youth_registration_variables_national.csv',  # IEC dashboard; filtered to MP in Stage 3
    # Census 2022 (all provinces kept in the file; Mpumalanga selected in Stage 3)
    'census_households':        DATASETS_DIR / 'Census2022_F18_F19_Combined_full.csv',
    'census_persons':           DATASETS_DIR / 'Census2022_raw' / 'Census2022sample_F21.csv',
    'census_geography':         DATASETS_DIR / 'Census2022_raw' / 'Census2022sample_F18.csv',
    # municipal performance: Auditor-General audit outcomes (National Treasury Municipal Money API)
    'agsa_audit_mp':            DATASETS_DIR / 'AGSA_audit_opinions' / 'audit_opinions_mpumalanga_raw.json',
    'agsa_audit_mp322_mp323':   DATASETS_DIR / 'AGSA_audit_opinions' / 'audit_opinions_mp322_mp323_raw.json',
}
missing = {k: str(v) for k, v in SOURCES.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Missing inputs: {missing}')
print(f'All {len(SOURCES)} sources found.')

In [ ]:
# =====================================================================================
# 1.3 Metadata (loaded before any data)
# -------------------------------------------------------------------------------------
# WHAT:  Loads the source registry (what each file is), the data dictionary, the
#        geography crosswalk and the Census codebook (what each Census code means).
# WHY:   The rules for using each dataset are visible at runtime, not hidden in our heads.
# =====================================================================================
source_registry     = pd.read_csv(SOURCES['metadata_source_registry'], dtype='string')
data_dictionary     = pd.read_csv(SOURCES['metadata_data_dictionary'], dtype='string')
geography_crosswalk = pd.read_csv(SOURCES['metadata_geo_crosswalk'], dtype='string')
census_codebook     = pd.read_csv(SOURCES['metadata_census_codebook'], keep_default_na=False)

assert source_registry['source_id'].is_unique
assert not census_codebook.duplicated(['variable', 'code']).any(), 'each census code must map to one label'
display(source_registry[['source_id', 'filename', 'status']])

In [ ]:
# =====================================================================================
# 1.4 Input manifest (fingerprints of the input files)
# -------------------------------------------------------------------------------------
# WHAT:  Records each main input's size, date and SHA-256 "fingerprint".
# WHY:   Reproducibility. If anyone changes a file, the fingerprint changes, so we can
#        prove which exact data produced our results.
# SAVED AS: derived/stage1_input_manifest.csv
# =====================================================================================
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_keys = ['baseline_lge_master', 'lge_2011_mp', 'lge_2016_mp', 'lge_2021_mp', 'npe_turnout_mp', 'voters_2026_report',
                 'census_households', 'agsa_audit_mp']
input_manifest = pd.DataFrame([
    {'input': k,
     'path': str(SOURCES[k].relative_to(PROJECT_ROOT)),
     'bytes': SOURCES[k].stat().st_size,
     'modified_utc': datetime.fromtimestamp(SOURCES[k].stat().st_mtime, tz=timezone.utc).isoformat(timespec='seconds'),
     'sha256': sha256_file(SOURCES[k])}
    for k in manifest_keys
])
input_manifest.to_csv(DERIVED_DIR / 'stage1_input_manifest.csv', index=False)
display(input_manifest)

### Raw ingestion

The tables below stay at source grain. Later stages create new objects and never overwrite these.

In [ ]:
# =====================================================================================
# 1.5 IEC results baseline (2000-2021)
# -------------------------------------------------------------------------------------
# WHAT:  Loads the team's combined IEC results file: one row per party x ballot x voting
#        district, which is why it has 721,327 rows.
# WHY:   It is our only source for 2000 and 2006. For 2011-2021 we read the original IEC
#        Mpumalanga files in Stage 2 and use the baseline to cross-check them.
# NOTE:  The source files "MP 2016.csv" and "MP 2021.csv" had their years swapped. The
#        baseline corrected this from each file's DateGenerated (column year_label_corrected).
# TAKES: about 1.5 minutes (487 MB file).
# =====================================================================================
baseline_raw = pd.read_csv(SOURCES['baseline_lge_master'], low_memory=False,
                           na_values=['', 'NA', 'N/A', 'null', 'None'])
print(f'baseline_raw: {baseline_raw.shape[0]:,} rows x {baseline_raw.shape[1]} columns')
display(baseline_raw.groupby(['election_year', 'source_filename']).agg(
    rows=('record_id', 'size'), provinces=('province', 'nunique'), municipalities=('municipality_code', 'nunique')))
assert baseline_raw.loc[baseline_raw['year_label_corrected'], 'source_filename'].isin(['MP 2016.csv', 'MP 2021.csv']).all()

In [ ]:
# =====================================================================================
# 1.6 2026 registered voters workbook (IEC)
# -------------------------------------------------------------------------------------
# WHAT:  Loads the Excel report as-is. It holds several small tables on one sheet
#        (by province, by age and gender, and by Mpumalanga municipality).
# WHY:   A cross-check for the 2026 registration figures used in Stage 3.
# NOTE:  Its label "MP314 - Emlahleni" is a typo for Emakhazeni (fixed in the crosswalk).
# =====================================================================================
voters_report_raw = pd.read_excel(SOURCES['voters_2026_report'], sheet_name=0, header=None, dtype=object)
print(f'voters_report_raw: {voters_report_raw.shape}')
display(voters_report_raw.head(15))

In [ ]:
# =====================================================================================
# 1.7 Census 2022 households (10% sample)
# -------------------------------------------------------------------------------------
# WHAT:  Loads every sampled household (1,338,295) with its municipality, living
#        conditions (coded) and household weight HH_WGT.
# WHY:   Provides the socio-economic context (internet, water, hunger...) in Stage 3.
# NOTE:  The file is kept whole (team decision). Mpumalanga (province code 8) is selected
#        in Stage 3. Official household counts use conventional dwellings only (codes 1-2).
# =====================================================================================
census_raw = pd.read_csv(SOURCES['census_households'], low_memory=False)
print(f'census_raw: {census_raw.shape[0]:,} households x {census_raw.shape[1]} columns (all provinces)')
assert len(census_raw) == 1_338_295 and census_raw['QID'].is_unique

conventional = census_raw['H01_QUARTERS'].isin([1, 2])
in_scope = census_raw['Province'].eq(SCOPE_CENSUS_PROVINCE_CODE)
print(f"Weighted households, all provinces (conventional dwellings): {census_raw.loc[conventional, 'HH_WGT'].sum():,.0f}  (published: 17.8 million)")
print(f"Weighted households, {SCOPE_PROVINCE}: {census_raw.loc[conventional & in_scope, 'HH_WGT'].sum():,.0f} "
      f"from {int((conventional & in_scope).sum()):,} sampled households")

In [ ]:
# =====================================================================================
# 1.8 Stage 1 summary
# =====================================================================================
stage1_summary = pd.DataFrame([
    {'object': 'baseline_raw',      'rows': len(baseline_raw),      'columns': baseline_raw.shape[1]},
    {'object': 'voters_report_raw', 'rows': len(voters_report_raw), 'columns': voters_report_raw.shape[1]},
    {'object': 'census_raw',        'rows': len(census_raw),        'columns': census_raw.shape[1]},
])
display(stage1_summary)
print('Stage 1 complete: metadata and raw inputs loaded; nothing transformed.')

## Stage 2: Mpumalanga election panel

Builds one table with **one row per voting district x election x ballot type**, then maps every voting district onto the **2021 municipal boundaries** so municipalities can be compared across elections.

| Election | Source |
|---|---|
| 2000, 2006 | `baseline_raw` (all provinces; Mpumalanga selected after mapping, so Bushbuckridge's 2000 districts, counted under Limpopo then, are kept) |
| 2011 | `DATASETS/LGE2011/MP_2011.csv` |
| 2016 | `DATASETS/LGE2016_MP/MP_2016.csv` |
| 2021 | `DATASETS/LGE2021_MP/MP_2021.csv` |

All ballot types (PR, Ward, DC 40%) are kept. Turnout later follows the IEC rule: the higher of the PR or Ward votes cast.

In [ ]:
# =====================================================================================
# 2.1 Read the original IEC result files for Mpumalanga (2011, 2016, 2021)
# -------------------------------------------------------------------------------------
# WHAT:  Reads each IEC file (one row per party x ballot x voting district) into one
#        table with clean column names and numbers.
# WHY:   These are the official detailed results. Every number is checked: the file's
#        generated date must match its election, and all rows must be Mpumalanga.
# OUTPUT: results_long (party-level rows)
# =====================================================================================
import io

RESULT_FILES = {2011: SOURCES['lge_2011_mp'], 2016: SOURCES['lge_2016_mp'], 2021: SOURCES['lge_2021_mp']}
BALLOT_NAMES = {'PR': 'PR', 'WARD': 'WARD', 'DC 40%': 'DC40', 'DMA DC 60%': 'DMA_DC60'}

def read_iec_results(path: Path, election_year: int) -> pd.DataFrame:
    """Read one IEC detailed-results file (handles UTF-16 and UTF-8) into the standard layout."""
    raw = path.read_bytes()
    encoding = 'utf-16' if raw[:2] in (b'\xff\xfe', b'\xfe\xff') else 'utf-8-sig'
    df = pd.read_csv(io.StringIO(raw.decode(encoding)), dtype=str)
    df.columns = df.columns.str.strip()
    generated = pd.to_datetime(df['DateGenerated'], format='%m/%d/%Y %I:%M:%S %p')
    assert set(generated.dt.year) == {election_year}, (path.name, set(generated.dt.year))   # right election?
    out = pd.DataFrame({
        'election_year': election_year,
        'province': df['Province'].str.strip(),
        'municipality_source': df['Municipality'].str.strip(),
        'ward': df['Ward'].str.replace('Ward', '', regex=False).str.strip(),
        'voting_district': df['VotingDistrict'].str.strip(),
        'ballot_type': df['BallotType'].str.strip().str.upper().map(BALLOT_NAMES),
        'party_name': df['PartyName'].str.strip(),
        'source_file': str(path.relative_to(PROJECT_ROOT)),
    })
    for src, dst in [('RegisteredVoters', 'registered_voters'), ('SpoiltVotes', 'spoilt_votes'), ('TotalValidVotes', 'party_valid_votes')]:
        out[dst] = pd.to_numeric(df[src], errors='raise').astype('int64')   # errors='raise': stop on any non-number
    assert out['ballot_type'].notna().all(), f'unknown ballot type in {path.name}'
    return out

results_long = pd.concat([read_iec_results(p, y) for y, p in RESULT_FILES.items()], ignore_index=True)
assert (results_long['province'] == SCOPE_PROVINCE).all()
display(results_long.groupby('election_year').agg(rows=('party_name', 'size'), municipalities=('municipality_source', 'nunique'),
                                                   voting_districts=('voting_district', 'nunique')))

In [ ]:
# =====================================================================================
# 2.2 Add 2000 and 2006 from the baseline (all provinces for now)
# -------------------------------------------------------------------------------------
# WHAT:  Converts the 2000/2006 baseline rows to the same layout as 2.1.
# WHY:   Boundaries changed a lot since 2000. Bushbuckridge (today MP325) was a
#        cross-border council counted under LIMPOPO in 2000. So we keep all provinces
#        here and select Mpumalanga AFTER mapping to 2021 boundaries (cell 2.5).
# MISSING VALUE: one 2000 row (Mdala Nature Reserve, a district management area) has
#        spoilt votes recorded as NULL at source. It is set to 0 and printed below.
# =====================================================================================
early = baseline_raw.loc[baseline_raw['election_year'].isin([2000, 2006])]
missing_spoilt = early['spoilt_votes'].isna()
print(f'Rows with spoilt votes recorded as NULL at source (set to 0): {int(missing_spoilt.sum())}')
display(early.loc[missing_spoilt, ['election_year', 'municipality_source', 'voting_district', 'ballot_type', 'registered_voters']])
early_long = pd.DataFrame({
    'election_year': early['election_year'].astype(int),
    'province': early['province'].str.strip(),
    'municipality_source': early['municipality_source'].str.strip(),
    'ward': early['ward_source'].fillna('').astype(str).str.strip(),   # district management areas have no wards
    'voting_district': early['voting_district'].astype(str).str.strip(),
    'ballot_type': early['ballot_type'].str.strip().str.upper().map(BALLOT_NAMES),
    'party_name': early['party_name'],
    'source_file': 'LGE_All_Sources_Master_721327_Rows.csv (' + early['source_filename'] + ')',
    'registered_voters': early['registered_voters'].astype('int64'),
    'spoilt_votes': early['spoilt_votes'].fillna(0).astype('int64'),
    'party_valid_votes': early['party_valid_votes'].astype('int64'),
})
assert early_long['ballot_type'].notna().all()
results_long = pd.concat([early_long, results_long], ignore_index=True)
print(f'results_long: {len(results_long):,} party rows across {results_long.election_year.nunique()} elections')

In [ ]:
# =====================================================================================
# 2.3 Collapse party rows: one row per voting district x election x ballot
# -------------------------------------------------------------------------------------
# WHAT:  Adds up each party's valid votes, and keeps registered voters and spoilt votes
#        (these repeat on every party row, so they must be identical: checked here).
#        votes_cast = valid votes + spoilt votes.
# WHY:   The model works per voting district, not per party.
# LOOK FOR: "conflicting repeated values: 0" (the data is internally consistent).
# NOTE:  A few districts have more votes than registered voters. That is normal: people
#        on another station's roll can vote with an MEC7 form.
# =====================================================================================
KEY = ['election_year', 'voting_district', 'ballot_type']
consistency = results_long.groupby(KEY).agg(
    registered_values=('registered_voters', 'nunique'), spoilt_values=('spoilt_votes', 'nunique'),
    municipalities=('municipality_source', 'nunique'), wards=('ward', 'nunique'))
problems = consistency[(consistency > 1).any(axis=1)]
print(f'Voting district x ballot groups with conflicting repeated values: {len(problems)} of {len(consistency):,}')

vd_panel = (results_long.groupby(KEY, as_index=False)
            .agg(province=('province', 'first'), municipality_source=('municipality_source', 'first'), ward=('ward', 'first'),
                 registered_voters=('registered_voters', 'max'), spoilt_votes=('spoilt_votes', 'max'),
                 valid_votes=('party_valid_votes', 'sum'), parties=('party_name', 'nunique'), source_file=('source_file', 'first')))
vd_panel['votes_cast'] = vd_panel['valid_votes'] + vd_panel['spoilt_votes']
vd_panel['municipality_code'] = vd_panel['municipality_source'].str.split(' - ').str[0].str.strip()
vd_panel['turnout_pct'] = 100 * vd_panel['votes_cast'] / vd_panel['registered_voters'].where(vd_panel['registered_voters'] > 0)
assert not vd_panel.duplicated(KEY).any()
print(f"vd_panel: {len(vd_panel):,} rows (all provinces for 2000/2006, Mpumalanga for 2011-2021)")

In [ ]:
# =====================================================================================
# 2.4 Cross-check: original IEC files vs the team baseline (Mpumalanga 2011-2021)
# -------------------------------------------------------------------------------------
# WHAT:  Compares voting districts, registered voters and votes cast between the two.
# WHY:   Two independent copies of the same results must agree exactly. If they do, we
#        can trust both, including the baseline's 2000/2006 data.
# LOOK FOR: "agree exactly". The cell stops with an error if they don't.
# =====================================================================================
check = []
for year in (2011, 2016, 2021):
    ours = vd_panel[(vd_panel.election_year == year) & (vd_panel.ballot_type == 'PR')]
    base = (baseline_raw[(baseline_raw.election_year == year) & (baseline_raw.ballot_type.str.upper() == 'PR')]
            .drop_duplicates('voting_district'))
    check.append({'election_year': year, 'vds_raw': len(ours), 'vds_baseline': len(base),
                  'registered_raw': ours.registered_voters.sum(), 'registered_baseline': int(base.registered_voters.sum()),
                  'votes_cast_raw': ours.votes_cast.sum(), 'votes_cast_baseline': int(base.ballot_votes_cast.sum())})
baseline_check = pd.DataFrame(check)
display(baseline_check)
assert (baseline_check.vds_raw == baseline_check.vds_baseline).all()
assert (baseline_check.registered_raw == baseline_check.registered_baseline).all()
assert (baseline_check.votes_cast_raw == baseline_check.votes_cast_baseline).all()
print('Original IEC files and baseline agree exactly for 2011, 2016 and 2021.')

### Mapping voting districts onto 2021 municipal boundaries

Boundaries changed between elections (for example MP322 Mbombela and MP323 Umjindi merged into MP326 City of Mbombela in 2016). Each earlier voting district is assigned to the 2021 Mpumalanga municipality it falls in:

1. **VD code match**: the voting district code also exists in 2021, so it takes that district's 2021 municipality.
2. **Municipality majority**: the code no longer exists (districts get re-drawn), so it takes the 2021 municipality that received most of its old municipality's matched voters. This rule is used only if **at least half** of the old municipality's voters matched into today's Mpumalanga by code.
3. **Outside Mpumalanga today**: neither rule points to a 2021 Mpumalanga municipality (other provinces, and areas that moved to Limpopo, such as CBLC3-5 Marble Hall, Groblersdal and Tubatse in 2000). These are dropped.

In [ ]:
# =====================================================================================
# 2.5 Map every voting district to its 2021 Mpumalanga municipality, then keep Mpumalanga
# -------------------------------------------------------------------------------------
# WHAT:  Applies the two rules above, then keeps only districts inside today's Mpumalanga.
# WHY:   So "MP326 in 2006" means the same land as "MP326 in 2021", and trends over time
#        are fair comparisons.
# LOOK FOR: (1) the share of voters mapped by exact code (should be ~99%);
#           (2) Bushbuckridge's 2000 districts arriving from "Limpopo";
#           (3) old Mpumalanga areas that now belong to Limpopo (dropped).
# =====================================================================================
vd_2021 = (vd_panel[vd_panel.election_year == 2021].sort_values('ballot_type')
           .groupby('voting_district').agg(municipality_code_2021=('municipality_code', 'first'),
                                           municipality_name_2021=('municipality_source', 'first'),
                                           province_2021=('province', 'first'), n_munis=('municipality_code', 'nunique')))
assert (vd_2021.n_munis == 1).all(), 'a 2021 voting district must sit in exactly one municipality'
vd_2021 = vd_2021.drop(columns='n_munis')

# one row per voting district per election, weighted by its registered voters (largest ballot)
vd_year = (vd_panel.groupby(['election_year', 'voting_district'], as_index=False)
           .agg(municipality_code=('municipality_code', 'first'), municipality_source=('municipality_source', 'first'),
                province=('province', 'first'), registered_voters=('registered_voters', 'max')))
vd_year = vd_year.merge(vd_2021, left_on='voting_district', right_index=True, how='left')
vd_year['boundary_method'] = np.where(vd_year['municipality_code_2021'].notna(), 'vd_code_match', None)

# Rule 2: old municipality -> the 2021 municipality that received most of its matched voters
matched = vd_year[vd_year.boundary_method.eq('vd_code_match')]
flows = matched.groupby(['election_year', 'municipality_code', 'municipality_code_2021'], as_index=False)['registered_voters'].sum()
flows['share'] = flows['registered_voters'] / flows.groupby(['election_year', 'municipality_code'])['registered_voters'].transform('sum')
majority = flows.sort_values('share', ascending=False).drop_duplicates(['election_year', 'municipality_code'])
# Guard: only use rule 2 if at least HALF of the old municipality's voters matched into today's
# Mpumalanga by code. Otherwise most of it now lies in another province (e.g. CBLC4 Groblersdal in
# 2000: only 0.6% matched), and its unmatched districts must not be pulled into Mpumalanga.
old_total = vd_year.groupby(['election_year', 'municipality_code'])['registered_voters'].sum()
matched_total = matched.groupby(['election_year', 'municipality_code'])['registered_voters'].sum()
matched_share = (matched_total / old_total).rename('matched_into_scope_share').reset_index()
majority = majority.merge(matched_share, on=['election_year', 'municipality_code'])
print('Old municipalities NOT given the majority rule (less than half their voters are in today\'s Mpumalanga):')
display(majority[majority['matched_into_scope_share'] < 0.5][['election_year', 'municipality_code', 'matched_into_scope_share']])
majority = majority[majority['matched_into_scope_share'] >= 0.5]
need = vd_year['boundary_method'].isna()
fill = vd_year.loc[need, ['election_year', 'municipality_code']].merge(
    majority[['election_year', 'municipality_code', 'municipality_code_2021']], how='left', on=['election_year', 'municipality_code'])
vd_year.loc[need, 'municipality_code_2021'] = fill['municipality_code_2021'].to_numpy()
vd_year.loc[need & vd_year['municipality_code_2021'].notna(), 'boundary_method'] = 'municipality_majority'
vd_year['boundary_method'] = vd_year['boundary_method'].fillna('outside_mpumalanga_2021')
names_2021 = vd_2021.drop_duplicates('municipality_code_2021').set_index('municipality_code_2021')
vd_year['municipality_name_2021'] = vd_year['municipality_code_2021'].map(names_2021['municipality_name_2021'])
vd_year['province_2021'] = vd_year['municipality_code_2021'].map(names_2021['province_2021'])

# Keep only districts that are in Mpumalanga today
vd_year_mp = vd_year[vd_year['province_2021'].eq(SCOPE_PROVINCE)].copy()
coverage = (vd_year_mp.groupby(['election_year', 'boundary_method'])['registered_voters'].sum()
            .unstack(fill_value=0).pipe(lambda t: 100 * t.div(t.sum(axis=1), axis=0)).round(2))
print('Share of registered voters in today\'s Mpumalanga, by mapping method (%):')
display(coverage)
print('Districts in today\'s Mpumalanga that were counted under ANOTHER province at the time:')
display(vd_year_mp[vd_year_mp.province != SCOPE_PROVINCE].groupby(['election_year', 'province', 'municipality_source'])['voting_district'].nunique())
left = vd_year[(vd_year.province == SCOPE_PROVINCE) & vd_year['province_2021'].ne(SCOPE_PROVINCE)]
print('Mpumalanga districts at the time that are NOT in Mpumalanga today (dropped):')
display(left.groupby(['election_year', 'municipality_source']).agg(voting_districts=('voting_district', 'nunique'),
                                                                   registered_voters=('registered_voters', 'sum')))

In [ ]:
# =====================================================================================
# 2.6 Municipality x election x ballot table (2021 boundaries, Mpumalanga only)
# -------------------------------------------------------------------------------------
# WHAT:  Attaches the 2021 municipality to every voting-district row, drops districts
#        outside today's Mpumalanga, and adds everything up per municipality.
# LOOK FOR: 17 municipalities in every election (2000-2021).
# OUTPUT: vd_panel (now Mpumalanga only) and muni_panel
# =====================================================================================
vd_panel = vd_panel.merge(vd_year[['election_year', 'voting_district', 'municipality_code_2021', 'municipality_name_2021',
                                   'province_2021', 'boundary_method']], on=['election_year', 'voting_district'], how='left')
vd_panel = vd_panel[vd_panel['province_2021'].eq(SCOPE_PROVINCE)].reset_index(drop=True)

muni_panel = (vd_panel.groupby(['province_2021', 'municipality_code_2021', 'municipality_name_2021', 'election_year', 'ballot_type'], as_index=False)
              .agg(registered_voters=('registered_voters', 'sum'), votes_cast=('votes_cast', 'sum'), valid_votes=('valid_votes', 'sum'),
                   spoilt_votes=('spoilt_votes', 'sum'), voting_districts=('voting_district', 'nunique'),
                   share_vd_code_match=('boundary_method', lambda s: (s == 'vd_code_match').mean())))
muni_panel['turnout_pct'] = 100 * muni_panel['votes_cast'] / muni_panel['registered_voters']
muni_panel['spoilt_pct'] = 100 * muni_panel['spoilt_votes'] / muni_panel['votes_cast']
assert not muni_panel.duplicated(['municipality_code_2021', 'election_year', 'ballot_type']).any()
munis_per_year = muni_panel[muni_panel.ballot_type == 'PR'].groupby('election_year')['municipality_code_2021'].nunique()
display(munis_per_year.rename('municipalities').to_frame())
assert (munis_per_year == SCOPE_MUNICIPALITIES).all()
print(f"vd_panel (Mpumalanga): {len(vd_panel):,} rows | muni_panel: {len(muni_panel):,} rows")

In [ ]:
# =====================================================================================
# 2.7 Reconciliation with the official IEC Mpumalanga turnout reports (2011, 2016, 2021)
# -------------------------------------------------------------------------------------
# WHAT:  Compares our registered voters and turnout per municipality with the IEC's own
#        published turnout reports.
# WHY:   Proves our processing reproduces official figures.
# HOW:   IEC turnout = max(PR, Ward) votes cast / (registered + MEC7 votes). MEC7 counts
#        are only in the official reports, so we add them here for the comparison.
# LOOK FOR: registered and turnout votes exact for (almost) every municipality.
#        Known exception: MP325 in 2021 (official report printed later than the results file).
# =====================================================================================
import csv, re

def turnout_iec_style(frame, group):
    """Turnout votes = the larger of PR and Ward votes cast in each voting district."""
    wide = frame.pivot_table(index=group + ['voting_district'], columns='ballot_type', values='votes_cast', aggfunc='sum')
    reg = frame.groupby(group + ['voting_district'])['registered_voters'].max()
    t = pd.DataFrame({'registered': reg, 'turnout_votes': wide[['PR', 'WARD']].max(axis=1)}).groupby(group).sum()
    t['turnout_pct'] = 100 * t['turnout_votes'] / t['registered']
    return t

def read_iec_turnout_report(path: Path) -> pd.DataFrame:
    """Pull the municipality lines out of the IEC's report-style CSV."""
    rows = []
    for fields in csv.reader(path.read_text(encoding='latin-1').splitlines()):
        vals = [f.strip() for f in fields if f.strip()]
        if vals and re.match(r'^[A-Z]{2,3}\d{0,3} - ', vals[0]) and len(vals) >= 5:
            rows.append({'municipality_code_2021': vals[0].split(' - ')[0], 'official_registered': int(vals[1].replace(',', '')),
                         'official_mec7': int(vals[2].replace(',', '')), 'official_turnout_votes': int(vals[3].replace(',', '')),
                         'official_turnout_pct': float(vals[4].rstrip('%'))})
    return pd.DataFrame(rows)

recon = []
for year in (2011, 2016, 2021):
    official = read_iec_turnout_report(SOURCES['iec_official_turnout_dir'] / f'MP_turnout_{year}.csv')
    ours = turnout_iec_style(vd_panel[vd_panel.election_year == year], ['municipality_code']).reset_index()
    ours = ours.rename(columns={'municipality_code': 'municipality_code_2021'})
    m = official.merge(ours, on='municipality_code_2021', how='outer', indicator=True)
    m['election_year'] = year
    recon.append(m)
recon_mp = pd.concat(recon, ignore_index=True)
recon_mp['registered_diff'] = recon_mp['registered'] - recon_mp['official_registered']
recon_mp['turnout_votes_diff'] = recon_mp['turnout_votes'] - recon_mp['official_turnout_votes']
recon_mp['turnout_pct_with_mec7'] = 100 * recon_mp['turnout_votes'] / (recon_mp['registered'] + recon_mp['official_mec7'])
recon_mp['pct_diff_pp'] = recon_mp['turnout_pct_with_mec7'] - recon_mp['official_turnout_pct']
summary = recon_mp.groupby('election_year').agg(
    municipalities=('municipality_code_2021', 'size'), matched=('_merge', lambda s: (s == 'both').sum()),
    registered_exact=('registered_diff', lambda s: (s == 0).sum()), turnout_votes_exact=('turnout_votes_diff', lambda s: (s == 0).sum()),
    max_abs_pct_diff_pp=('pct_diff_pp', lambda s: s.abs().max()))
display(summary)
display(recon_mp.loc[(recon_mp.registered_diff != 0) | (recon_mp.turnout_votes_diff != 0),
                     ['election_year', 'municipality_code_2021', 'official_registered', 'registered', 'official_turnout_votes', 'turnout_votes', 'pct_diff_pp']])

In [ ]:
# =====================================================================================
# 2.8 National/provincial election turnout per municipality (2004-2024)
# -------------------------------------------------------------------------------------
# WHAT:  Loads the team's table of provincial-election turnout per Mpumalanga
#        municipality (compiled from the IEC's municipal turnout reports, file column
#        "src") and puts it on 2021 boundaries:
#          - MP322 Mbombela + MP323 Umjindi are added together as MP326 City of Mbombela
#          - 2004 cross-border areas (CBLC2-5, CBDMA3-4) and district management areas
#            are not part of any 2021 Mpumalanga municipality, so they are dropped
# WHY:   A national/provincial election sits between two local elections. 2024 is the
#        most recent turnout signal before the 2026 local election. National turnout is
#        always higher than local turnout, so the model uses it as a relative signal.
# LOOK FOR: which years are present. 2019 is NOT in the file yet (see printed warning),
#        so the 2021 local election can only use the 2014 provincial election.
# NOTE:  turnout = votes / registered voters (column "reg"). One row (2014 MP302) has a
#        different "pop" figure; "reg" is used throughout, consistent with other years.
# =====================================================================================
npe_raw = pd.read_csv(SOURCES['npe_turnout_mp'])
TO_2021_CODE = {'MP322': 'MP326', 'MP323': 'MP326'}      # 2016 merger
npe_raw['municipality_code_2021'] = npe_raw['code'].replace(TO_2021_CODE)
mp_codes_2021 = set(muni_panel['municipality_code_2021'])
dropped = npe_raw[~npe_raw['municipality_code_2021'].isin(mp_codes_2021)]
print('Rows not part of a 2021 Mpumalanga municipality (dropped):')
display(dropped[['year', 'code', 'name', 'reg', 'votes', 'note']])

npe_muni = (npe_raw[npe_raw['municipality_code_2021'].isin(mp_codes_2021)]
            .groupby(['year', 'municipality_code_2021'])[['reg', 'votes']].sum())
npe_muni['npe_turnout_pct'] = 100 * npe_muni['votes'] / npe_muni['reg']
npe_turnout = npe_muni['npe_turnout_pct'].unstack('year')          # rows = municipality, columns = election year
print('Provincial-election turnout per municipality (2021 boundaries):')
display(npe_turnout.round(1))
expected_npe_years = {2004, 2009, 2014, 2019, 2024}
missing_years = sorted(expected_npe_years - set(npe_turnout.columns))
if missing_years:
    print(f'WARNING: provincial elections missing from the file: {missing_years}. Features fall back to the latest earlier election.')

In [ ]:
# =====================================================================================
# 2.9 Save Stage 2 outputs
# -------------------------------------------------------------------------------------
# SAVES: derived/election_vd_panel.parquet  (voting district x election x ballot, Mpumalanga)
#        derived/vd_to_2021_municipality_map.csv
#        derived/municipality_election_panel_2021_boundaries.csv
#        derived/reconciliation_mp_official_turnout.csv
#        derived/npe_turnout_municipality.csv (provincial elections, 2021 boundaries)
# =====================================================================================
vd_panel.to_parquet(DERIVED_DIR / 'election_vd_panel.parquet', index=False)
vd_year_mp.to_csv(DERIVED_DIR / 'vd_to_2021_municipality_map.csv', index=False)
muni_panel.to_csv(DERIVED_DIR / 'municipality_election_panel_2021_boundaries.csv', index=False)
recon_mp.drop(columns='_merge').to_csv(DERIVED_DIR / 'reconciliation_mp_official_turnout.csv', index=False)
npe_muni.round(3).to_csv(DERIVED_DIR / 'npe_turnout_municipality.csv')
for f in ['election_vd_panel.parquet', 'vd_to_2021_municipality_map.csv', 'municipality_election_panel_2021_boundaries.csv',
          'reconciliation_mp_official_turnout.csv', 'npe_turnout_municipality.csv']:
    print(f'{f:52s} {(DERIVED_DIR / f).stat().st_size:>12,} bytes')
print('Stage 2 complete.')

## Stage 3: Census 2022 indicators, registration denominators and municipal performance

Three municipal-level building blocks for Mpumalanga's 17 municipalities:

1. **Census 2022 living conditions**, following the rules in `metadata/README.md`: conventional dwellings only, always weighted, and Unspecified / Not applicable answers left out of numerator and denominator (each indicator carries `*_answered_pct`).
2. **Registration denominators**: adult South African citizens aged to each voters' roll date using birth year and month. Registration rate = registered voters / eligible citizens.
3. **Municipal performance**: Auditor-General audit outcomes per financial year (National Treasury Municipal Money data).

Census geography type: 1 = Urban, 2 = Traditional, 3 = Farms (Stats SA Census 2022 metadata).

In [ ]:
# =====================================================================================
# 3.1 Census household indicators per Mpumalanga municipality
# -------------------------------------------------------------------------------------
# WHAT:  For each municipality, the % of households with internet, piped water,
#        electricity, weekly refuse removal, adult hunger and so on.
# HOW:   (1) only conventional dwellings (matches official counts);
#        (2) every household counts by its weight HH_WGT (it represents ~10 households);
#        (3) "Unspecified" / "Not applicable" answers are left out, NOT counted as "no".
#        The *_answered_pct column shows how many households answered each question.
# LOOK FOR: 17 municipalities; answer rates (the internet/goods/hunger questions were
#        skipped by some households, which Stats SA flags as a reliability issue).
# =====================================================================================
hh = census_raw.loc[census_raw['H01_QUARTERS'].isin([1, 2]) & census_raw['Province'].eq(SCOPE_CENSUS_PROVINCE_CODE)].copy()

# Codes that mean "no answer" for each variable, taken from the codebook rather than typed by hand
no_answer = (census_codebook[census_codebook['label'].str.contains('Unspecified|Not applicable', case=False)]
             .groupby('variable')['code'].apply(set).to_dict())

# name: (variable, codes counted as "yes", extra codes to leave out of the denominator)
HOUSEHOLD_INDICATORS = {
    'internet_any':         ('H13_INTERNET_ACCESS', {1, 2, 3, 4, 5, 6, 7, 8}, set()),
    'internet_at_home':     ('H13_INTERNET_ACCESS', {1}, set()),
    'internet_mobile_only': ('H13_INTERNET_ACCESS', {2}, set()),
    'cellphone':            ('H12_CELLPHONE', {1}, set()),
    'computer':             ('H12_COMPUTER', {1}, set()),
    'motor_car':            ('H12_MOTOR_CAR', {1}, set()),
    'piped_water_in_yard':  ('H05_WATERPIPED', {1, 2}, set()),
    'electricity_lighting': ('H10_ENERGY_LIGHTING', {1}, set()),
    'flush_toilet':         ('H08_TOILET', {1, 2}, set()),
    'refuse_weekly':        ('H11_REFUSE', {1}, set()),
    'informal_dwelling':    ('H02_MAINDWELLING', {8, 9}, set()),
    'rdp_house':            ('H04_RDP', {1}, {3}),                  # 3 = do not know
    'adult_hunger':         ('A4_ADULT_HUNGER', {3, 4, 5}, {6}),    # sometimes/often/always; 6 = no adult
    'female_headed':        ('DERH_HHSEX', {2}, set()),
}

def weighted_indicator(frame, variable, yes_codes, extra_excluded):
    """Weighted % of answering households with a 'yes' code, plus the weighted % that answered."""
    excluded = no_answer.get(variable, set()) | extra_excluded
    answered = ~frame[variable].isin(excluded)
    w = frame['HH_WGT']
    g = pd.DataFrame({'Municipality': frame['Municipality'], 'w_all': w,
                      'w_answered': w.where(answered, 0), 'w_yes': w.where(answered & frame[variable].isin(yes_codes), 0)})
    s = g.groupby('Municipality')[['w_all', 'w_answered', 'w_yes']].sum()
    return 100 * s['w_yes'] / s['w_answered'], 100 * s['w_answered'] / s['w_all']

census_muni = hh.groupby('Municipality').agg(district_code=('District', 'first'),
                                             households=('HH_WGT', 'sum'), sample_households=('QID', 'size'))
for name, (variable, yes, extra) in HOUSEHOLD_INDICATORS.items():
    census_muni[f'{name}_pct'], census_muni[f'{name}_answered_pct'] = weighted_indicator(hh, variable, yes, extra)

geo = hh.pivot_table(index='Municipality', columns='Geo_type', values='HH_WGT', aggfunc='sum', fill_value=0)
geo = 100 * geo.div(geo.sum(axis=1), axis=0)
census_muni[['urban_pct', 'traditional_pct', 'farms_pct']] = geo.reindex(columns=[1, 2, 3], fill_value=0).to_numpy()
census_muni['mean_household_size'] = hh.assign(x=hh['DERH_HSIZE'] * hh['HH_WGT']).groupby('Municipality')['x'].sum() / census_muni['households']

census_muni.index.name = 'municipality_code'
assert len(census_muni) == SCOPE_MUNICIPALITIES
print(f"{len(census_muni)} municipalities | weighted households {census_muni['households'].sum():,.0f}")
display(census_muni[[c for c in census_muni if c.endswith('_pct') and not c.endswith('answered_pct')]].round(1))

In [ ]:
# =====================================================================================
# 3.2 Eligible voters: adult South African citizens per municipality
# -------------------------------------------------------------------------------------
# WHAT:  Counts citizens aged 18+ (and 18-29, 30+) in each municipality, using their
#        age ON THE DATE of each voters' roll (1 Nov 2021 and 23 Sep 2026). The age is
#        worked out from birth year and month, not the 2022 age, so someone who was
#        14 in 2022 correctly counts as 18 in 2026.
# WHY:   The denominator for the registration rate (Part B).
# NOTE:  Uses the Census person file (weights PERS_WGT). People who died or moved after
#        Feb 2022 can't be removed, so the 2026 count is somewhat too high (see note below 3.3).
# =====================================================================================
persons = pd.read_csv(SOURCES['census_persons'],
                      usecols=['QID', 'P03_YEAR', 'P03_MONTH', 'P04_AGE', 'P10_CITIZENSHIP', 'PERS_WGT'])
geo_f18 = pd.read_csv(SOURCES['census_geography'], usecols=['QID', 'Province', 'Municipality'])
persons = persons.merge(geo_f18, on='QID', how='left', validate='many_to_one')
assert persons['Municipality'].notna().all()
persons = persons[persons['Province'].eq(SCOPE_CENSUS_PROVINCE_CODE)]

# Age on a given date from year and month of birth; unknown month (99) is treated as mid-year
birth_month = persons['P03_MONTH'].where(persons['P03_MONTH'].between(1, 12), 7)
def age_on(year, month):
    return year - persons['P03_YEAR'] - (birth_month > month).astype(int)

ROLL_DATES = {2021: (2021, 11),   # 2021 LGE voters' roll (election 1 Nov 2021)
              2026: (2026, 9)}    # IEC registration dashboard, as of 23 Sep 2026
citizen = persons['P10_CITIZENSHIP'].eq(101)          # 101 = South Africa
print(f"Persons with citizenship unspecified: {persons['P10_CITIZENSHIP'].eq(999).mean():.2%} (left out)")

eligible = []
for roll_year, (y, m) in ROLL_DATES.items():
    age = age_on(y, m)
    for label, lo, hi in [('eligible_18plus', 18, 200), ('eligible_18_29', 18, 29), ('eligible_30plus', 30, 200)]:
        s = persons.loc[citizen & age.between(lo, hi)].groupby('Municipality')['PERS_WGT'].sum().rename(f'{label}_{roll_year}')
        eligible.append(s)
eligible = pd.concat(eligible, axis=1).fillna(0)
eligible.index.name = 'municipality_code'
display(eligible.sum().round(0).rename(f'{SCOPE_PROVINCE} total').to_frame())

In [ ]:
# =====================================================================================
# 3.3 Registration rates, 2021 and 2026 (overall, youth 18-29, older 30+)
# -------------------------------------------------------------------------------------
# WHAT:  registration rate = registered voters / eligible citizens (from 3.2).
#        2021 registered voters come from the election results (Stage 2);
#        2026 registered voters come from the IEC dashboard (23 Sep 2026), by age.
# WHY:   This is Part B, the REGISTRATION GAP, including the youth gap
#        (older rate minus youth rate, in percentage points).
# LOOK FOR: the Mpumalanga totals line and the per-municipality table.
# =====================================================================================
reg_2021 = (muni_panel[(muni_panel.election_year == 2021) & (muni_panel.ballot_type == 'PR')]
            .set_index('municipality_code_2021')['registered_voters'].rename('registered_2021'))

youth_raw = pd.read_csv(SOURCES['youth_registration_2026'], skiprows=2)   # 2 comment lines above the header
youth_raw = youth_raw[youth_raw['province'].eq(SCOPE_PROVINCE)]
youth_raw['youth_18_29'] = youth_raw['age_group'].isin(['18-19', '20-29'])
reg_2026 = youth_raw.groupby('municipality_code').agg(
    registered_2026=('registered_voters', 'sum'),
    registered_2026_18_29=('registered_voters', lambda s: s[youth_raw.loc[s.index, 'youth_18_29']].sum()))
print(f"2026 Mpumalanga roll (dashboard): {reg_2026['registered_2026'].sum():,} | IEC workbook total: 2,169,679")
assert set(reg_2026.index) == set(census_muni.index) == set(reg_2021.index), 'municipality codes must line up across sources'

registration = eligible.join(reg_2021).join(reg_2026)
registration['registration_rate_2021_pct'] = 100 * registration['registered_2021'] / registration['eligible_18plus_2021']
registration['registration_rate_2026_pct'] = 100 * registration['registered_2026'] / registration['eligible_18plus_2026']
registration['youth_registration_rate_2026_pct'] = 100 * registration['registered_2026_18_29'] / registration['eligible_18_29_2026']
registration['older_registration_rate_2026_pct'] = (100 * (registration['registered_2026'] - registration['registered_2026_18_29'])
                                                    / registration['eligible_30plus_2026'])
registration['youth_registration_gap_pp'] = registration['older_registration_rate_2026_pct'] - registration['youth_registration_rate_2026_pct']
registration['youth_share_of_roll_2026_pct'] = 100 * registration['registered_2026_18_29'] / registration['registered_2026']
registration['roll_growth_2021_2026_pct'] = 100 * (registration['registered_2026'] / registration['registered_2021'] - 1)

tot = registration.sum()
print(f"{SCOPE_PROVINCE} registration rate: 2021 {100 * tot.registered_2021 / tot.eligible_18plus_2021:.1f}% | "
      f"2026 {100 * tot.registered_2026 / tot.eligible_18plus_2026:.1f}% | "
      f"2026 youth 18-29 {100 * tot.registered_2026_18_29 / tot.eligible_18_29_2026:.1f}% vs 30+ "
      f"{100 * (tot.registered_2026 - tot.registered_2026_18_29) / tot.eligible_30plus_2026:.1f}%")
display(registration[['registration_rate_2021_pct', 'registration_rate_2026_pct', 'youth_registration_rate_2026_pct',
                      'older_registration_rate_2026_pct', 'youth_registration_gap_pp', 'youth_share_of_roll_2026_pct']]
        .sort_values('youth_registration_gap_pp', ascending=False).round(1))

**How to read these rates.** The denominators come from a 10% Census sample taken in February 2022, aged forward by birth date. They are estimates:

- **Deaths after February 2022 are not removed.** The 2026 eligible count is therefore too high, mostly among older adults. As a result, 2026 registration rates are *lower bounds*, and the youth-vs-older gap is *understated*, so the real gap is larger. The youth (18-29) rate is the most reliable, because youth mortality is low.
- Migration after 2022 is not captured, and people in institutions and the homeless are outside the Census sample.

In [ ]:
# =====================================================================================
# 3.4 Municipal performance: Auditor-General audit outcomes
# -------------------------------------------------------------------------------------
# WHAT:  Loads each municipality's audit opinion for every financial year 2011-2024
#        (the year a financial year ENDS, in June) and turns it into a score:
#          4 = clean audit (unqualified, no findings)
#          3 = unqualified with findings ("emphasis of matter")
#          2 = qualified          1 = adverse          0 = disclaimer (worst)
#        "Outstanding" (statements not submitted) has no score and is flagged.
# WHY:   The whiteboard's "Municipal scores (performance)": does weak governance go
#        with lower participation? Used in Parts A and C.
# SOURCE: National Treasury Municipal Money API (cube audit_opinions), downloaded
#        24 Sep 2026; each row links to the Auditor-General's report.
# MP326: City of Mbombela was formed in 2016 from MP322 (Mbombela) and MP323 (Umjindi).
#        For 2011-2015 its score is the two predecessors' scores weighted by their 2011
#        registered voters.
# =====================================================================================
import json

OPINION_SCORE = {'unqualified': 4, 'unqualified_emphasis_of_matter': 3, 'qualified': 2, 'adverse': 1, 'disclaimer': 0}

def load_audit(path):
    d = pd.DataFrame(json.loads(path.read_text(encoding='utf-8'))['data'])
    return d.rename(columns={'demarcation.code': 'municipality_code', 'demarcation.label': 'municipality_label',
                             'financial_year_end.year': 'financial_year_end', 'opinion.code': 'opinion_code',
                             'opinion.label': 'opinion', 'opinion.report_url': 'report_url'})

audit_raw = pd.concat([load_audit(SOURCES['agsa_audit_mp']), load_audit(SOURCES['agsa_audit_mp322_mp323'])], ignore_index=True)
audit_raw['audit_score'] = audit_raw['opinion_code'].map(OPINION_SCORE)     # 'outstanding' -> NaN
audit_raw['outstanding'] = audit_raw['opinion_code'].eq('outstanding')
print('Opinions:', audit_raw['opinion'].value_counts().to_dict())

# MP326 before it existed: weighted average of MP322 and MP323 (weights = 2011 registered voters)
w = (vd_panel[(vd_panel.election_year == 2011) & (vd_panel.ballot_type == 'PR') & vd_panel.municipality_code.isin(['MP322', 'MP323'])]
     .groupby('municipality_code')['registered_voters'].sum())
pred = audit_raw[audit_raw.municipality_code.isin(['MP322', 'MP323'])].pivot_table(index='financial_year_end', columns='municipality_code', values='audit_score')
mp326_early = (pred * w).sum(axis=1) / w.sum()
audit_scores = audit_raw[~audit_raw.municipality_code.isin(['MP322', 'MP323'])].pivot_table(
    index='municipality_code', columns='financial_year_end', values='audit_score')
for fy, v in mp326_early.items():
    if pd.isna(audit_scores.loc['MP326'].get(fy, np.nan)):
        audit_scores.loc['MP326', fy] = v
audit_scores = audit_scores.sort_index(axis=1)
assert len(audit_scores) == SCOPE_MUNICIPALITIES
print(f'Predecessor weights for MP326 (2011 registered voters): {w.to_dict()}')
display(audit_scores.round(1))

In [ ]:
# =====================================================================================
# 3.5 Save Stage 3 outputs
# -------------------------------------------------------------------------------------
# SAVES: derived/census2022_municipality_indicators.csv  (living conditions)
#        derived/registration_denominators_2021_2026.csv  (eligible citizens, rates, youth gap)
#        derived/agsa_audit_scores.csv                      (audit score per municipality x year)
#        derived/agsa_audit_opinions_long.csv               (raw opinions with report links)
# =====================================================================================
census_muni.round(3).to_csv(DERIVED_DIR / 'census2022_municipality_indicators.csv')
registration.round(3).to_csv(DERIVED_DIR / 'registration_denominators_2021_2026.csv')
audit_scores.round(3).to_csv(DERIVED_DIR / 'agsa_audit_scores.csv')
audit_raw.to_csv(DERIVED_DIR / 'agsa_audit_opinions_long.csv', index=False)
for f in ['census2022_municipality_indicators.csv', 'registration_denominators_2021_2026.csv', 'agsa_audit_scores.csv', 'agsa_audit_opinions_long.csv']:
    print(f'{f:45s} {(DERIVED_DIR / f).stat().st_size:>10,} bytes')
print('Stage 3 complete.')

## Stage 4: Exploratory analysis and feature engineering

Team decisions applied from here on:

| Decision | Choice |
|---|---|
| Scope | **Mpumalanga**, 17 municipalities (2021 boundaries) |
| Youth | ages **18-29** (matches the IEC roll bands) |
| History used | **all elections 2000-2021** (2000 serves as history only; turnout is predicted for 2006, 2011, 2016, 2021 and 2026) |
| Turnout | **IEC rule**: max(PR, Ward) votes cast / registered voters, per voting district |

Every figure is saved as a PNG in `derived/figures/` for the slides, and each one is followed by the table of the numbers it shows.

In [ ]:
# =====================================================================================
# 4.1 Turnout per voting district and election (IEC rule)
# -------------------------------------------------------------------------------------
# WHAT:  Works out turnout for every Mpumalanga voting district in every election.
#        IEC rule = the HIGHER of the PR-ballot or Ward-ballot votes cast, divided by
#        registered voters.
# WHY:   Turnout is what the model predicts (Part A).
# OUTPUT: vd_turnout (district x election); province_turnout (Mpumalanga total by year);
#         muni_turnout (municipality x election)
# =====================================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ELECTIONS = [2000, 2006, 2011, 2016, 2021]
pr = vd_panel[vd_panel.ballot_type == 'PR'].set_index(['election_year', 'voting_district'])
ward_votes = vd_panel[vd_panel.ballot_type == 'WARD'].set_index(['election_year', 'voting_district'])['votes_cast']

vd_turnout = pr[['province_2021', 'municipality_code_2021', 'municipality_name_2021', 'registered_voters', 'spoilt_votes', 'votes_cast']].copy()
vd_turnout['turnout_votes'] = np.fmax(vd_turnout['votes_cast'], ward_votes.reindex(vd_turnout.index))   # IEC rule
vd_turnout = vd_turnout[vd_turnout['registered_voters'] > 0].reset_index()
vd_turnout['turnout_pct'] = 100 * vd_turnout['turnout_votes'] / vd_turnout['registered_voters']
vd_turnout['spoilt_pct'] = 100 * vd_turnout['spoilt_votes'] / vd_turnout['votes_cast'].where(vd_turnout['votes_cast'] > 0)

def turnout_by(frame, keys):
    """Add up votes and registered voters for a group, THEN divide.
    (Averaging percentages would give a tiny district the same weight as a big one.)"""
    t = frame.groupby(keys)[['registered_voters', 'turnout_votes']].sum()
    t['turnout_pct'] = 100 * t['turnout_votes'] / t['registered_voters']
    return t

province_turnout = turnout_by(vd_turnout, ['election_year'])
muni_turnout = turnout_by(vd_turnout, ['province_2021', 'municipality_code_2021', 'municipality_name_2021', 'election_year']).reset_index()
muni_turnout['short_name'] = muni_turnout['municipality_name_2021'].str.split(' - ').str[1]
print(f"vd_turnout: {len(vd_turnout):,} voting district x election rows")
display(province_turnout.round(2))

In [ ]:
# =====================================================================================
# 4.2 Chart style shared by every figure
# -------------------------------------------------------------------------------------
# WHAT:  One look for all charts: light background, thin lines, faint gridlines, blue for
#        the data, grey for reference lines, orange only as a second series.
# SAVES: every chart as a PNG in derived/figures/ (ready for the slides).
# =====================================================================================
FIG_DIR = DERIVED_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
for old in FIG_DIR.glob('*.png'):      # remove figures from earlier runs (e.g. the all-province version)
    old.unlink()
INK, INK_2, MUTED, GRID, BASE, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
SERIES_1, SERIES_2, REFERENCE = '#2a78d6', '#eb6834', '#898781'
DIVERGING = {'positive': '#2a78d6', 'negative': '#e34948'}
ORDINAL = ['#e34948', '#f0a0a0', '#d9d8d2', '#86b6ef', '#1c5cab']   # audit score 0 (worst) .. 4 (clean)
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.family': ['Segoe UI', 'DejaVu Sans'], 'font.size': 10, 'text.color': INK,
    'axes.edgecolor': BASE, 'axes.labelcolor': INK_2, 'axes.titlecolor': INK, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'lines.linewidth': 2, 'legend.frameon': False,
})
def save(fig, name):
    """Save the figure as a high-resolution PNG for the slides, then show it in the notebook."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# =====================================================================================
# 4.3 Figure 1: Mpumalanga turnout, 2000-2021
# -------------------------------------------------------------------------------------
# WHAT:  One line: Mpumalanga's local-election turnout in each election.
# WHY:   The opening fact of our story: turnout dropped sharply in 2021.
# SAVED AS: derived/figures/fig1_mpumalanga_turnout.png
# =====================================================================================
fig, ax = plt.subplots(figsize=(8, 4))
x, y = province_turnout.index, province_turnout['turnout_pct']
ax.plot(x, y, color=SERIES_1, marker='o', markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
for xi, yi in zip(x, y):   # write the % above each point
    ax.annotate(f'{yi:.1f}%', (xi, yi), textcoords='offset points', xytext=(0, 10), ha='center', color=INK_2)
ax.set_ylim(0, 70); ax.set_xticks(ELECTIONS)      # y starts at 0 so the drop isn't exaggerated
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title(f'Mpumalanga turnout fell from {y.loc[2016]:.1f}% in 2016 to {y.loc[2021]:.1f}% in 2021')
ax.set_ylabel('Turnout of registered voters')
ax.text(0, -0.18, 'Turnout = max(PR, Ward) votes cast / registered voters, 2021 boundaries. Source: IEC results; team calculation.',
        transform=ax.transAxes, color=MUTED, fontsize=8)
save(fig, 'fig1_mpumalanga_turnout')

In [ ]:
# =====================================================================================
# 4.4 Figure 2: turnout by municipality (17 small charts)
# -------------------------------------------------------------------------------------
# WHAT:  One small chart per municipality. BLUE = that municipality, GREY = Mpumalanga as
#        a whole (the same grey line in every panel, as a reference).
# HOW TO READ: blue above grey = the municipality voted more than the province; below = less.
# SAVED AS: derived/figures/fig2_municipal_turnout.png (all values in the table below)
# =====================================================================================
munis = muni_turnout.sort_values('municipality_code_2021')[['municipality_code_2021', 'short_name']].drop_duplicates()
fig, axes = plt.subplots(5, 4, figsize=(11, 12), sharex=True, sharey=True)
for ax, (code_, name) in zip(axes.flat, munis.itertuples(index=False)):
    ax.plot(province_turnout.index, province_turnout['turnout_pct'], color=REFERENCE, linewidth=1.2)
    p = muni_turnout[muni_turnout.municipality_code_2021 == code_].set_index('election_year')['turnout_pct']
    ax.plot(p.index, p.values, color=SERIES_1, marker='o', markersize=4)
    ax.annotate(f'{p.iloc[-1]:.0f}%', (p.index[-1], p.iloc[-1]), textcoords='offset points', xytext=(7, 0), ha='left', va='center', color=INK_2, fontsize=8)
    ax.set_title(f'{code_} {name}', fontsize=9); ax.set_ylim(25, 75); ax.set_xlim(1998, 2025); ax.set_xticks([2000, 2011, 2021])
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
for ax in axes.flat[len(munis):]:
    ax.set_visible(False)                         # 17 municipalities, 20 slots
axes.flat[0].legend(handles=[plt.Line2D([], [], color=SERIES_1, lw=2, label='Municipality'),
                             plt.Line2D([], [], color=REFERENCE, lw=1.2, label='Mpumalanga')], loc='lower left', fontsize=7)
fig.suptitle('Turnout by municipality, 2000-2021 (grey: Mpumalanga)', x=0.02, ha='left', fontweight='bold')
fig.tight_layout()
save(fig, 'fig2_municipal_turnout')
display(muni_turnout.pivot_table(index=['municipality_code_2021', 'short_name'], columns='election_year', values='turnout_pct').round(1))

In [ ]:
# =====================================================================================
# 4.5 Figure 3: change in turnout per municipality, 2016 -> 2021
# -------------------------------------------------------------------------------------
# WHAT:  One bar per municipality: percentage points (pp) gained or lost between 2016 and
#        2021, sorted from the biggest drop.
# WHY:   Shows where the 2021 collapse was worst.
# SAVED AS: derived/figures/fig3_turnout_change_2016_2021.png
# =====================================================================================
m = muni_turnout.pivot_table(index=['municipality_code_2021', 'short_name'], columns='election_year', values='turnout_pct')
m['change_2016_2021_pp'] = m[2021] - m[2016]
m = m.sort_values('change_2016_2021_pp', ascending=False)
labels = [f'{c} {n}' for c, n in m.index]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(labels, m['change_2016_2021_pp'], color=SERIES_1, height=0.7)
for i, v in enumerate(m['change_2016_2021_pp']):
    ax.annotate(f'{v:+.1f}', (v, i), xytext=(-4 if v < 0 else 4, 0), textcoords='offset points',
                ha='right' if v < 0 else 'left', va='center', color=INK_2, fontsize=8)
ax.axvline(0, color=INK_2, linewidth=1); ax.grid(axis='y', visible=False)
ax.set_xlim(min(-25, m['change_2016_2021_pp'].min() - 3), 3)
ax.set_xlabel('Change in turnout, 2016 to 2021 (percentage points)')
ax.set_title(f"Turnout fell in {(m['change_2016_2021_pp'] < 0).sum()} of {len(m)} municipalities between 2016 and 2021")
save(fig, 'fig3_turnout_change_2016_2021')
display(m.round(1))

In [ ]:
# =====================================================================================
# 4.6 Figure 4: 2026 registration, youth (18-29) vs 30+, per municipality
# -------------------------------------------------------------------------------------
# WHAT:  For each municipality, two dots joined by a line: the % of 18-29-year-old
#        citizens registered (blue) and the % of 30+ citizens registered (orange).
# WHY:   The YOUTH REGISTRATION GAP from our problem statement (Part B). The longer the
#        line, the bigger the gap. Sorted with the biggest gap at the top.
# SAVED AS: derived/figures/fig4_youth_vs_older_registration_2026.png
# =====================================================================================
r = registration.sort_values('youth_registration_gap_pp')
names = muni_turnout.drop_duplicates('municipality_code_2021').set_index('municipality_code_2021')['short_name']
labels = [f'{c} {names[c]}' for c in r.index]
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.hlines(labels, r['youth_registration_rate_2026_pct'], r['older_registration_rate_2026_pct'], color=GRID, linewidth=3)
ax.scatter(r['youth_registration_rate_2026_pct'], labels, s=60, color=SERIES_1, edgecolor=SURFACE, linewidth=1.5, zorder=3, label='Ages 18-29')
ax.scatter(r['older_registration_rate_2026_pct'], labels, s=60, color=SERIES_2, edgecolor=SURFACE, linewidth=1.5, zorder=3, label='Ages 30+')
for i, (c, row) in enumerate(r.iterrows()):
    ax.annotate(f"{row['youth_registration_gap_pp']:.0f} pp", (row['older_registration_rate_2026_pct'], i), xytext=(8, 0),
                textcoords='offset points', va='center', color=INK_2, fontsize=8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); ax.grid(axis='y', visible=False)
ax.set_xlabel('Share of eligible citizens registered (2026)')
ax.set_title(f"Youth registration trails 30+ registration in {(r['youth_registration_gap_pp'] > 0).sum()} of {len(r)} municipalities")
ax.legend(loc='lower right')
ax.text(0, -0.12, 'Registered voters (IEC dashboard, 23 Sep 2026) / SA citizens of that age (Census 2022, aged forward by birth date).',
        transform=ax.transAxes, color=MUTED, fontsize=7)
save(fig, 'fig4_youth_vs_older_registration_2026')
display(r[['youth_registration_rate_2026_pct', 'older_registration_rate_2026_pct', 'youth_registration_gap_pp', 'youth_share_of_roll_2026_pct']].round(1))

In [ ]:
# =====================================================================================
# 4.7 Figure 5: municipal performance, Auditor-General audit outcomes 2011-2024
# -------------------------------------------------------------------------------------
# WHAT:  A grid: rows = municipalities, columns = financial years. Each cell shows the
#        audit opinion (C = clean, U = unqualified with findings, Q = qualified,
#        A = adverse, D = disclaimer, - = outstanding) and is coloured from red (worst)
#        to blue (best).
# WHY:   The "municipal scores (performance)" part of our plan, at a glance.
# SAVED AS: derived/figures/fig5_audit_outcomes.png
# =====================================================================================
from matplotlib.colors import ListedColormap, BoundaryNorm
LETTER = {4: 'C', 3: 'U', 2: 'Q', 1: 'A', 0: 'D'}
grid = audit_scores.loc[sorted(audit_scores.index)]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.imshow(grid.to_numpy(dtype=float), cmap=ListedColormap(ORDINAL), norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], 5), aspect='auto')
outstanding = audit_raw[audit_raw.outstanding].set_index(['municipality_code', 'financial_year_end']).index
for i, code_ in enumerate(grid.index):
    for j, fy in enumerate(grid.columns):
        v = grid.loc[code_, fy]
        txt = '-' if (code_, fy) in outstanding else ('' if pd.isna(v) else LETTER.get(int(round(v)), ''))
        if code_ == 'MP326' and fy <= 2015 and not pd.isna(v):
            txt = f'{v:.1f}'                          # blended MP322/MP323 score
        ax.text(j, i, txt, ha='center', va='center', fontsize=8, color=SURFACE if (not pd.isna(v) and (v <= 0.5 or v >= 3.5)) else INK)
ax.set_xticks(range(len(grid.columns)), [str(c) for c in grid.columns], rotation=0, fontsize=8)
ax.set_yticks(range(len(grid.index)), [f'{c} {names[c]}' for c in grid.index], fontsize=8)
ax.grid(False)
ax.set_xlabel('Financial year ending June')
ax.set_title('Auditor-General audit outcomes, Mpumalanga municipalities')
ax.text(0, -0.1, 'C clean  U unqualified with findings  Q qualified  A adverse  D disclaimer  - outstanding  blank = no record. '
                 'MP326 2011-2015: voter-weighted MP322/MP323 score. Source: National Treasury Municipal Money (AGSA).',
        transform=ax.transAxes, color=MUTED, fontsize=7)
save(fig, 'fig5_audit_outcomes')

In [ ]:
# =====================================================================================
# 4.8 Figure 6: which conditions go with higher or lower 2021 turnout
# -------------------------------------------------------------------------------------
# WHAT:  For each indicator, a number between -1 and +1 (Spearman rank correlation):
#        did municipalities with MORE of it have HIGHER (+, blue) or LOWER (-, red)
#        turnout in 2021?
# CAUTION: only 17 municipalities, so these are indicative. With n = 17, a correlation
#        needs to be about +/-0.48 or more to be statistically significant (5% level).
#        This shows ASSOCIATION ONLY, not cause.
# SAVED AS: derived/figures/fig6_correlation_turnout_2021.png
# =====================================================================================
t21 = muni_turnout[muni_turnout.election_year == 2021].set_index('municipality_code_2021')['turnout_pct']
context_cols = ['internet_any_pct', 'internet_at_home_pct', 'computer_pct', 'motor_car_pct', 'piped_water_in_yard_pct',
                'electricity_lighting_pct', 'flush_toilet_pct', 'refuse_weekly_pct', 'informal_dwelling_pct', 'rdp_house_pct',
                'adult_hunger_pct', 'female_headed_pct', 'urban_pct', 'traditional_pct', 'farms_pct', 'mean_household_size']
audit_before_2021 = audit_scores[[2018, 2019, 2020]].mean(axis=1).rename('audit_score_fy2018_2020')
extra = registration[['registration_rate_2021_pct', 'youth_share_of_roll_2026_pct']].join(audit_before_2021)
corr = census_muni[context_cols].join(extra).corrwith(t21, method='spearman').sort_values().rename('spearman_rho').to_frame()
fig, ax = plt.subplots(figsize=(7, 6.5))
colors = [DIVERGING['positive'] if v >= 0 else DIVERGING['negative'] for v in corr['spearman_rho']]
ax.barh(corr.index.str.replace('_pct', '').str.replace('_', ' '), corr['spearman_rho'], color=colors, height=0.7)
for i, v in enumerate(corr['spearman_rho']):
    ax.annotate(f'{v:+.2f}', (v, i), xytext=(4 if v >= 0 else -4, 0), textcoords='offset points',
                ha='left' if v >= 0 else 'right', va='center', color=INK_2, fontsize=8)
for edge in (-0.48, 0.48):
    ax.axvline(edge, color=GRID, linewidth=1)
ax.axvline(0, color=INK_2, linewidth=1); ax.set_xlim(-1, 1); ax.grid(axis='y', visible=False)
ax.set_xlabel('Spearman rank correlation with 2021 turnout (17 municipalities; |rho| > 0.48 = significant)')
ax.set_title('Conditions linked to 2021 turnout in Mpumalanga\n(association, not cause)')
save(fig, 'fig6_correlation_turnout_2021')
display(corr.round(3))

### Feature engineering

One row per **voting district x target election**. Every feature must pass the leakage test: *would this value be known before the election being predicted?* Anything from the target election's own count (votes cast, spoilt votes, turnout) is excluded.

| Feature group | Built from | Known before the election? |
|---|---|---|
| Previous turnout of the voting district (1 and 2 elections back) and its change | earlier elections | yes |
| Previous turnout of its municipality and of Mpumalanga | earlier elections | yes |
| Size of the district at the previous election | earlier roll | yes |
| Growth of the municipality's voters' roll since the previous election | roll certified before election day (2026: IEC dashboard, 23 Sep 2026) | yes |
| Previous spoilt-ballot rate | earlier elections | yes |
| Municipal audit score: latest financial year, and 3-year average | Auditor-General, financial years ending at least a year before the election | yes (not available before FY2011, so missing for the 2006 and 2011 targets) |
| Latest national/provincial election turnout of the municipality, and how many years before | IEC provincial-election reports 2004-2024 (2019 not yet in the file) | yes (only elections held before the target year) |
| Census 2022 municipal context | Census 2022 | yes for 2026; for 2006-2021 it is a later snapshot used as slow-moving context (stated limitation) |

The youth share of the roll exists only for 2026, so it is **not** a model feature. It enters Part C (priority list) instead.

In [ ]:
# =====================================================================================
# 4.9 Feature table: one row per voting district x election we want to predict
# -------------------------------------------------------------------------------------
# WHAT:  For each target election (2006, 2011, 2016, 2021, and 2026 for the forecast),
#        every voting district gets FEATURES known BEFORE that election, e.g.:
#          vd_turnout_lag1      = the district's turnout at the PREVIOUS election
#          vd_turnout_lag2      = its turnout two elections back
#          muni_turnout_lag1    = its municipality's previous turnout
#          muni_roll_growth_pct = how much the municipality's voters' roll grew
#          muni_audit_score_last / _3yr = municipal audit performance before the election
#          c22_...              = Census 2022 conditions of the municipality
# WHY:   A model may only use information available before election day. Using the
#        election's own result would be cheating ("leakage").
# AUDIT TIMING: an audit for the financial year ending June Y is published around
#        Nov-Dec Y. So for an election in year T we use financial years up to T-1
#        (2026: the latest available, FY2024).
# LOOK FOR: the summary table. vd_history_pct = % of districts that also existed at the
#        previous election. New or renumbered districts fall back on municipal history.
# =====================================================================================
def lagged(frame, value_col, key_cols):
    """Wide table: one row per district (or municipality), one column per election."""
    return frame.pivot_table(index=key_cols, columns='election_year', values=value_col)

vd_t = lagged(vd_turnout, 'turnout_pct', 'voting_district')
vd_reg = lagged(vd_turnout, 'registered_voters', 'voting_district')
vd_spoilt = lagged(vd_turnout, 'spoilt_pct', 'voting_district')
mu_t = lagged(muni_turnout, 'turnout_pct', 'municipality_code_2021')
mu_reg = lagged(muni_turnout, 'registered_voters', 'municipality_code_2021')
mu_reg[2026] = registration['registered_2026']          # 2026 roll (IEC dashboard)

TARGETS = [2006, 2011, 2016, 2021, 2026]
PREVIOUS = {2006: 2000, 2011: 2006, 2016: 2011, 2021: 2016, 2026: 2021}   # "the election before"

def audit_features(t):
    """Latest audit score and 3-year average using financial years up to t-1."""
    years = [fy for fy in audit_scores.columns if fy <= t - 1]
    if not years:
        return pd.Series(np.nan, index=audit_scores.index), pd.Series(np.nan, index=audit_scores.index)
    last = audit_scores[years].ffill(axis=1).iloc[:, -1]
    avg3 = audit_scores[years[-3:]].mean(axis=1)
    return last, avg3

def npe_features(t):
    """Most recent provincial-election turnout held BEFORE year t, per municipality, and how many years ago.
    If a municipality has no value in that year (e.g. MP325 in 2004), its latest earlier value is used."""
    years = [y for y in npe_turnout.columns if y < t]
    if not years:
        return pd.Series(np.nan, index=npe_turnout.index), pd.Series(np.nan, index=npe_turnout.index)
    sub = npe_turnout[years]
    last_year = sub.notna().mul(sub.columns.to_numpy()).replace(0, np.nan).max(axis=1)
    return sub.ffill(axis=1).iloc[:, -1], t - last_year

rows = []
for t in TARGETS:
    p1 = PREVIOUS[t]; p2 = PREVIOUS.get(p1)             # previous election, and the one before that
    base_year = t if t != 2026 else 2021                # 2026 districts = the 2021 districts
    base = vd_turnout[vd_turnout.election_year == base_year][['voting_district', 'province_2021', 'municipality_code_2021', 'municipality_name_2021']].copy()
    base['target_year'] = t
    vd = base['voting_district']; mu = base['municipality_code_2021']
    # --- features: all from EARLIER elections (p1, p2), the pre-election roll, or earlier audits
    base['vd_turnout_lag1'] = vd.map(vd_t[p1])
    base['vd_turnout_lag2'] = vd.map(vd_t[p2]) if p2 else np.nan
    base['vd_turnout_change_lag'] = base['vd_turnout_lag1'] - base['vd_turnout_lag2']
    base['vd_history_available'] = base['vd_turnout_lag1'].notna()
    base['vd_registered_lag1_log'] = np.log1p(vd.map(vd_reg[p1]))      # district size (log scale)
    base['vd_spoilt_pct_lag1'] = vd.map(vd_spoilt[p1])
    base['muni_turnout_lag1'] = mu.map(mu_t[p1])
    base['muni_turnout_lag2'] = mu.map(mu_t[p2]) if p2 else np.nan
    base['province_turnout_lag1'] = province_turnout.loc[p1, 'turnout_pct']
    base['muni_roll_growth_pct'] = 100 * (mu.map(mu_reg[t]) / mu.map(mu_reg[p1]) - 1)
    base['years_since_previous'] = t - p1
    last, avg3 = audit_features(t)
    base['muni_audit_score_last'] = mu.map(last)
    base['muni_audit_score_3yr'] = mu.map(avg3)
    npe_last, npe_age = npe_features(t)
    base['muni_npe_turnout_last'] = mu.map(npe_last)             # latest provincial-election turnout before t
    base['muni_npe_years_before'] = mu.map(npe_age)              # how many years before t that election was
    if t != 2026:   # --- the TARGET (the answer), only for elections that have already happened
        base['turnout_pct'] = vd.map(vd_t[t])
        base['registered_voters'] = vd.map(vd_reg[t])
    rows.append(base)
features_vd = pd.concat(rows, ignore_index=True)

census_features = census_muni[context_cols].add_prefix('c22_')
features_vd = features_vd.merge(census_features, left_on='municipality_code_2021', right_index=True, how='left')

summary = features_vd.groupby('target_year').agg(rows=('voting_district', 'size'),
                                                 vd_history_pct=('vd_history_available', 'mean'),
                                                 audit_available_pct=('muni_audit_score_last', lambda s: s.notna().mean()),
                                                 target_available=('turnout_pct', lambda s: s.notna().mean()))
summary[['vd_history_pct', 'audit_available_pct']] = (100 * summary[['vd_history_pct', 'audit_available_pct']]).round(1)
display(summary)

In [ ]:
# =====================================================================================
# 4.10 Leakage check and feature dictionary
# -------------------------------------------------------------------------------------
# WHAT:  Automatic tests proving no feature contains the answer, plus a table describing
#        every feature (source, known before the election?, % missing).
# WHY:   "Leakage" = using information from the election you are predicting. It makes a
#        model look brilliant in testing and useless in reality. Judges look for this.
# TESTS: (1) no outcome column (votes, turnout...) is a feature;
#        (2) for 2021 rows, "previous turnout" equals the 2016 value, NOT the 2021 value;
#        (3) audit features for 2021 use financial years <= 2020 only.
#        If a test fails, the cell stops with an error.
# =====================================================================================
FEATURES = [c for c in features_vd.columns if c.startswith(('vd_', 'muni_', 'province_turnout', 'years_', 'c22_')) and c != 'vd_history_available']
TARGET = 'turnout_pct'
forbidden = {'turnout_votes', 'votes_cast', 'spoilt_votes', 'turnout_pct', 'registered_voters'}
assert not forbidden & set(FEATURES), 'a same-election outcome leaked into the features'

chk = features_vd[(features_vd.target_year == 2021) & features_vd.vd_turnout_lag1.notna()]
assert np.allclose(chk['vd_turnout_lag1'], chk['voting_district'].map(vd_t[2016]))
assert not np.allclose(chk['vd_turnout_lag1'], chk['turnout_pct'])
a21 = features_vd[features_vd.target_year == 2021].drop_duplicates('municipality_code_2021').set_index('municipality_code_2021')['muni_audit_score_last']
assert np.allclose(a21, audit_scores.loc[a21.index, [fy for fy in audit_scores.columns if fy <= 2020]].ffill(axis=1).iloc[:, -1], equal_nan=True)
# (4) provincial-election features always come from an election held BEFORE the target year
assert (features_vd['muni_npe_years_before'].dropna() > 0).all(), 'a provincial election in or after the target year leaked in'

feature_dictionary = pd.DataFrame({'feature': FEATURES})
feature_dictionary['source'] = np.select(
    [feature_dictionary.feature.str.startswith('c22_'), feature_dictionary.feature.eq('muni_roll_growth_pct'),
     feature_dictionary.feature.str.startswith('muni_audit'), feature_dictionary.feature.str.startswith('muni_npe')],
    ['Census 2022 (Stage 3)', 'IEC voters roll (Stage 2; 2026: IEC dashboard)', 'Auditor-General via Municipal Money (Stage 3)',
     'IEC provincial-election turnout reports (Stage 2, cell 2.8)'],
    'IEC results, earlier elections (Stage 2)')
feature_dictionary['known_before_election'] = np.select(
    [feature_dictionary.feature.str.startswith('c22_'), feature_dictionary.feature.str.startswith('muni_audit')],
    ['yes for 2026; later snapshot for 2006-2021', 'yes (financial years up to the year before the election)'], 'yes')
feature_dictionary['missing_pct'] = [round(100 * features_vd.loc[features_vd.target_year < 2026, f].isna().mean(), 1) for f in FEATURES]
display(feature_dictionary)
print(f'{len(FEATURES)} features passed the leakage checks.')

In [ ]:
# =====================================================================================
# 4.11 Municipality table (for Parts B and C) and save everything
# -------------------------------------------------------------------------------------
# WHAT:  One row per municipality: turnout 2000-2021, registration rates 2021/2026, the
#        youth gap, youth share of the roll, audit scores and Census context. Also
#        PARTICIPATION = registration rate x turnout = share of ALL eligible citizens
#        who actually voted in 2021.
# SAVES: derived/features_vd.parquet, derived/features_municipality.csv,
#        derived/feature_dictionary.csv
# =====================================================================================
muni_features = (muni_turnout.pivot_table(index=['province_2021', 'municipality_code_2021', 'municipality_name_2021'],
                                          columns='election_year', values='turnout_pct')
                 .add_prefix('turnout_').reset_index().set_index('municipality_code_2021'))
muni_features['turnout_change_2016_2021_pp'] = muni_features['turnout_2021'] - muni_features['turnout_2016']
latest_fy = max(audit_scores.columns)
muni_features['audit_score_latest'] = audit_scores.ffill(axis=1)[latest_fy]
muni_features['audit_score_avg_last3'] = audit_scores[sorted(audit_scores.columns)[-3:]].mean(axis=1)
muni_features = (muni_features.join(registration[['registration_rate_2021_pct', 'registration_rate_2026_pct',
                                                  'youth_registration_rate_2026_pct', 'older_registration_rate_2026_pct',
                                                  'youth_registration_gap_pp', 'youth_share_of_roll_2026_pct',
                                                  'roll_growth_2021_2026_pct', 'registered_2026', 'eligible_18plus_2026']])
                             .join(census_features))
muni_features['participation_2021_pct'] = muni_features['registration_rate_2021_pct'] * muni_features['turnout_2021'] / 100

features_vd.to_parquet(DERIVED_DIR / 'features_vd.parquet', index=False)
muni_features.round(3).to_csv(DERIVED_DIR / 'features_municipality.csv')
feature_dictionary.to_csv(DERIVED_DIR / 'feature_dictionary.csv', index=False)
for f in ['features_vd.parquet', 'features_municipality.csv', 'feature_dictionary.csv']:
    print(f'{f:30s} {(DERIVED_DIR / f).stat().st_size:>10,} bytes')
print('Figures:', sorted(p.name for p in FIG_DIR.glob('*.png')))
print('Stage 4 complete.')